In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("asefjamilajwad/car-crash-dataset-ccd")

print("Path to dataset files:", path)

c:\Users\visha\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 7.61G/7.61G [21:46<00:00, 6.26MB/s]  

Extracting files...


Path to dataset files: C:\Users\visha\.cache\kagglehub\datasets\asefjamilajwad\car-crash-dataset-ccd\versions\2


In [2]:

import os
import shutil
import glob
import cv2
from pathlib import Path
from ultralytics import YOLO

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
# The path from your kagglehub output
dataset_source = "C:/Users/visha/.cache/kagglehub/datasets/asefjamilajwad/car-crash-dataset-ccd/versions/2"
train_root = "/content/train_data"

# Clean up any old folders
if os.path.exists(train_root): shutil.rmtree(train_root)
os.makedirs(f"{train_root}/accident", exist_ok=True)
os.makedirs(f"{train_root}/normal", exist_ok=True)

# Find all images
all_images = glob.glob(f"{dataset_source}/**/*.jpg", recursive=True)
print(f"Total images found: {len(all_images)}")

# Logic: Split frames by index to create two classes
for img_path in all_images[:20000]: # Using 20k images for a strong, accurate model
    filename = Path(img_path).stem
    try:
        # File format is usually 'C_000001_42' where 42 is the frame number
        frame_num = int(filename.split('_')[-1])
        
        if frame_num > 35: # Impact frames
            shutil.copy(img_path, f"{train_root}/accident/{filename}.jpg")
        else: # Safe driving frames
            shutil.copy(img_path, f"{train_root}/normal/{filename}.jpg")
    except:
        continue

print(f"✅ Organized: {len( os.listdir(train_root+'/accident'))} Accidents")
print(f"✅ Organized: {len(os.listdir(train_root+'/normal'))} Normal driving")

Total images found: 75000
✅ Organized: 6000 Accidents
✅ Organized: 14000 Normal driving


In [5]:
# 1. Load the pre-trained classification model
model = YOLO('yolov8s-cls.pt') 

# 2. Start training using the T4 GPU
# imgsz=640: Standard high-res for dashcam detail
# epochs=20: Enough for high accuracy on this large dataset
model.train(
    data=train_root, 
    epochs=2, 
    imgsz=640, 
    batch=32, 
    device=0, 
    name='sjec_accident_model'
)

print("\n--- ✅ TRAINING COMPLETE ---")

New https://pypi.org/project/ultralytics/8.4.24 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.218  Python-3.10.11 torch-2.2.2+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [1]:
import torch
print("--- GPU STATUS CHECK ---")
cuda_ready = torch.cuda.is_available()
print(f"Is GPU Available? {cuda_ready}")

if cuda_ready:
    print(f"Found GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU NOT detected. Training will be slow on CPU.")

--- GPU STATUS CHECK ---
Is GPU Available? True
Found GPU: NVIDIA GeForce GTX 1650


In [5]:
import os
import shutil
import glob
from pathlib import Path

# 1. FIXED PATHS FOR WINDOWS
dataset_source = r"C:\Users\visha\.cache\kagglehub\datasets\asefjamilajwad\car-crash-dataset-ccd\versions\2"
# Creating the training folder on your Desktop for easy access
train_root = os.path.join(os.path.expanduser("~"), "Desktop", "train_data")

# 2. CLEAN & CREATE FOLDERS
if os.path.exists(train_root): 
    shutil.rmtree(train_root)
os.makedirs(os.path.join(train_root, "accident"), exist_ok=True)
os.makedirs(os.path.join(train_root, "normal"), exist_ok=True)

# 3. FIND IMAGES
all_images = glob.glob(f"{dataset_source}/**/*.jpg", recursive=True)
print(f"Total images found in Kaggle cache: {len(all_images)}")

# 4. SORTING LOGIC
# Increased to 30k images for better accuracy since you have a GPU
for img_path in all_images[:30000]: 
    filename = Path(img_path).stem
    try:
        # CCD format: 'C_000001_42' -> 42 is the frame
        frame_num = int(filename.split('_')[-1])
        
        if frame_num > 35: 
            shutil.copy(img_path, os.path.join(train_root, "accident", f"{filename}.jpg"))
        else: 
            shutil.copy(img_path, os.path.join(train_root, "normal", f"{filename}.jpg"))
    except:
        continue

print(f"✅ Organized at: {train_root}")
print(f"✅ Accidents: {len(os.listdir(os.path.join(train_root, 'accident')))}")
print(f"✅ Normal: {len(os.listdir(os.path.join(train_root, 'normal')))}")

Total images found in Kaggle cache: 75000
✅ Organized at: C:\Users\visha\Desktop\train_data
✅ Accidents: 9000
✅ Normal: 21000


In [7]:
from ultralytics import YOLO

def main():
    # Load the Small model (Pre-trained)
    model = YOLO('yolov8s-cls.pt') 

    # Training on your GTX 1650
    # imgsz=416: Saves VRAM on your 1650 while keeping high accuracy
    # batch=16: Perfect for 4GB of video memory
    model.train(
        data=train_root, 
        epochs=3, 
        imgsz=416, 
        batch=16, 
        device=0, # This uses your CUDA GPU
        workers=2, # Optimized for local Windows CPUs
        name='sjec_accident_model'
    )

    print("\n--- ✅ TRAINING FINISHED! ---")
    print("Your model is saved in the 'runs' folder in this directory.")

if __name__ == '__main__':
    main()

New https://pypi.org/project/ultralytics/8.4.24 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.218  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\visha\Desktop\train_data, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=sjec_accident_model2

In [ ]:
import cv2
from ultralytics import YOLO

# 1. SETUP PATHS
# Use 'r' for Windows paths. Update these to your actual file locations.
model_path = r"runs/classify/sjec_accident_model2/weights/best.pt"
image_path = r"C:/Users/visha/Desktop/test_image.jpg" 

# 2. LOAD MODEL
model = YOLO(model_path)

# 3. RUN PREDICTION (Using your GTX 1650)
results = model.predict(source=image_path, device=0, conf=0.25)

# 4. PROCESS AND DISPLAY
for result in results:
    # Get the top prediction
    probs = result.probs
    label_index = probs.top1
    label_name = result.names[label_index]
    confidence = probs.top1conf.item()

    print(f"Prediction: {label_name.upper()} | Confidence: {confidence:.2%}")

    # Draw result on the image using OpenCV
    img = cv2.imread(image_path)
    text = f"{label_name.upper()} ({confidence:.1%})"
    color = (0, 0, 255) if label_name == "accident" else (0, 255, 0)
    
    cv2.putText(img, text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv2.imshow("Accident Detection Test", img)
    cv2.waitKey(0) # Press any key to close
    cv2.destroyAllWindows()